# ut 谱 × meta-gradient 金标准
**回答的问题：ut 作为"下游贡献度代理"有没有价值。**

meta-grad（一阶）：∂L_task/∂w(t) ∝ −⟨∇L_task, ∇L_diff(t)⟩，是该 t 对下游任务真实增益的直接度量。ut 谱与 meta-grad 谱的 Spearman/Pearson 相关性越高，ut 越可信。

支持对多个 checkpoint 批量测量并汇总相关性演化；测量结果逐 ckpt 缓存为 npz，论文拼图（挑 ckpt × 挑 k）直接读缓存、免重测。

**ut 定稿形式（风险调整后的相对贡献）：**
**μ(t) = (D(t) − k·σ_D(t)) / (E[L₀(t)] + δ)**，δ = 0.1×median(L̄₀)
- 分子（减法）管存亡：σ_D 大到把 D̄ 扣穿零的 t 被压制（不可复现的贡献不算数），D̄ 与 σ_D 同量纲，减法合法；
- 分母（除法）管排序：除以无条件难度 L̄₀，存活 t 按占无条件难度的比例排剂量，跨 t/跨任务可比；
- 小 t 处 D̄、σ_D 都→0，μ 自动趋零，无需人工压制；
- k ∈ {0, 1, 2, 3} 四档扫描，k=0 退化为 drel（基准）。

可靠性设计：两轴数据劈半（防共享数据虚增相关）、双侧 CRN 重复、相关系数带子集 bootstrap 95% CI。

meta 轴归一化：主量为单位化内积 ⟨g_task, g_diff/‖g_diff‖⟩——"沿该 t 方向走单位步长的 task loss 变化"，跨 t 可比；
裸内积保留在 mg['meta_raw'] 作诊断。meta 轴 32 个 t 点 × 4 个不同图像子集 × 2 噪声（loss_var 实验证明
方差由图像异质性主导、ε 臂≈0）；逐点误差带 = 跨子集 std，相关 CI = 对图像子集 bootstrap。

**论文拼图**：最后一个 cell，挑指定的 checkpoints（行）× 指定的 k 值（列）出 n_ckpts × n_k 网格图，
PNG + PDF 双格式，直接进论文。k 不限于四档——μ 由缓存的原始统计量现算，换 k 免重测。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把两个路径都算成绝对路径 ----
VISUAL = os.path.abspath('.')                                  # 内核启动时 cwd 就是 visual/
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import probe_core as pc

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CFG 用 run 目录里 train 时 dump 的完整 config，口径与训练严格一致；CKPTS 按 iter 排序。
# 旧 run 兼容：build_model 会自动剔除 dump config 里已退役的机制键（旧 t_weight
# 控制器 / 旧蒙块），旧 ckpt 直接可测；UNet 仪器的旧 run 需加 UNET_FIX（见下）。
# 可靠性设计（v3，预算按方差来源重分配：Var_x0 主导、Var_ε≈0）：
#   - 两个轴用不同数据：96 张图，前 32 给 ut 谱、后 64 给 meta-grad——
#     共享数据会共同原因虚增相关，劈开是防线；
#   - ut 轴：4 噪声 CRN × 32 图（噪声臂方差≈0，预算给图像）；
#   - meta 轴：32 个 t 点 × 4 个不同图像子集 × 2 噪声，跨子集 std 即
#     逐点测量误差（图里红色误差带）；g_task 用全部 64 图一次反传；
#   - 相关 bootstrap 对图像子集重采样——CI 覆盖'换批图曲线就变'的真不确定性。
import glob, re
RUN_DIR = 'work_dirs/eden/fcn_diffae'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]   # run 目录里 dump 的 config
UNET_FIX = None   # UNet 仪器的旧 run 改为 dict(denoiser_type='unet')；DiT run 保持 None
CKPTS = sorted(glob.glob(f'{RUN_DIR}/iter_*.pth'),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = 'work_dirs/offline_probe/ut_meta'
NUM_IMAGES = 96       # 前 32 给 ut 谱，后 64 给 meta-grad
N_NOISE_UT = 4        # ut 轴噪声重复数（CRN）
N_T_MG = 32           # meta 轴 t 点数（等距）
N_SUBSETS = 4         # meta 轴每 t 的图像子集数（64 图 ÷ 4 = 16 图/子集）
N_NOISE_MG = 2        # meta 轴每子集噪声数（ε 臂方差≈0，2 个足够）
SEED = 0
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG, cfg_overrides=UNET_FIX)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
rows = []
# 数据分配：ut 与 meta-grad 用不同图像（防共享数据虚增相关）
N_UT = 32
def _sub(b, sl):
    return {'cond': b['cond'][sl], 'x_diffae': b['x_diffae'][sl],
            'xis': b['xis'][sl], 'dsis': b['dsis'][sl]}
batch_ut = _sub(batch, slice(0, N_UT))
batch_mg = _sub(batch, slice(N_UT, NUM_IMAGES))
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt, cfg_overrides=UNET_FIX)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    ut = pc.measure_ut_spectrum(model, batch_ut, n_noise=N_NOISE_UT, seed=SEED)
    mg = pc.measure_meta_grad_spectrum(model, batch_mg, n_t=N_T_MG,
                                       n_subsets=N_SUBSETS, n_noise=N_NOISE_MG,
                                       seed=SEED + 1)
    pc.save_probe_cache(os.path.join(OUT, f'{tag}_cache.npz'), ut, mg)  # 拼图免重测
    fig, corrs = pc.plot_ut_vs_meta(
        ut, mg, title=tag,
        save_path=os.path.join(OUT, f"{tag}_ut_vs_meta.png"))
    rows.append((tag, corrs))
    s = ' | '.join(f"{k}: ρ={v[0]:.3f}{'CI[%0.2f,%0.2f]' % v[3] if v[3][0] == v[3][0] else ''}"
                   for k, v in corrs.items())
    print(f"{tag}: {s}")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 汇总：四个 k 档与 meta-grad 的相关性随训练演化
import matplotlib.pyplot as plt
if len(rows) > 1:
    tags = [r[0] for r in rows]
    keys = list(rows[0][1].keys())
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for k in keys:
        axes[0].plot(tags, [r[1][k][0] for r in rows], 'o-', label=k)
        axes[1].plot(tags, [r[1][k][2] for r in rows], 's-', label=k)
    for ax, name in zip(axes, ['Spearman ρ', 'Pearson r']):
        ax.axhline(0, color='k', ls='--', lw=0.8)
        ax.set_ylim(-1, 1); ax.grid(alpha=0.3); ax.legend(fontsize=8)
        ax.set_title(f'ut ~ meta-grad {name} across checkpoints')
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.savefig(os.path.join(OUT, 'summary.png'), dpi=150, bbox_inches='tight')
    fig

## 论文拼图：挑 ckpt × 挑 k
从缓存 npz 读测量结果（免重测），出 **n_ckpts × n_k** 网格图：行 = checkpoint，列 = k 值；
每格 μ(t) 蓝线（左轴）+ meta-grad 红线（右轴，可关），左上角标 Spearman ρ。
同列共享 μ 量程（跨 ckpt 可比）。`MONTAGE_KS` 不限于 {0,1,2,3}——μ 由原始统计量现算，任意 k 都行。

In [ ]:
# ---- 拼图配置：改成要展示的 ckpt tag 与 k 值 ----
MONTAGE_CKPTS = [os.path.splitext(os.path.basename(c))[0] for c in CKPTS]  # 缺省全部；挑几个就写成 ['iter_4000', 'iter_24000', ...]
MONTAGE_KS = [0, 1, 2, 3]     # 列：任意 k，换值免重测
X_AXIS = 'logsnr'             # 'logsnr'（论文视图）或 't'
WITH_META = True              # 每格是否叠 meta-grad 红线

panels = []
for tag in MONTAGE_CKPTS:
    path = os.path.join(OUT, f'{tag}_cache.npz')
    assert os.path.exists(path), f'缺缓存 {path}——先跑上面的测量 cell（或把该 tag 从 MONTAGE_CKPTS 去掉）'
    ut, mg = pc.load_probe_cache(path)
    panels.append({'tag': tag, 'ut': ut, 'mg': mg})

fig = pc.plot_ut_montage(panels, MONTAGE_KS, x_axis=X_AXIS, with_meta=WITH_META,
                         save_path=os.path.join(OUT, 'montage.png'))
fig.savefig(os.path.join(OUT, 'montage.pdf'), bbox_inches='tight')   # 矢量版进论文
fig